# Class 4 — Isolation Forest (ML Detector)

**Deliverable:** ML anomaly model
**Date:** July 19, 2026

Goal: replace the hand-written rule with an unsupervised model that can learn how
features interact. Two attempts are recorded below — the first one failed, and the
comparison between them is the most useful result in the whole project.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())

In [ ]:
# Feature 1 - amount z-score: how unusual is this amount vs the average
mean_amount = df['Amount'].mean()
std_amount  = df['Amount'].std()
df['amount_zscore'] = (df['Amount'] - mean_amount) / std_amount

# Feature 2 - hour of day: Time is seconds since first transaction, not clock time
df['hour'] = (df['Time'] / 3600) % 24

# Feature 3 - is_night: midnight to 6am
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 6)).astype(int)

print(df[['Amount', 'amount_zscore', 'hour', 'is_night']].head())

In [ ]:
def evaluate(flag_series, label='detector'):
    """Precision/recall for a boolean flag column against the true Class label."""
    caught  = df[(flag_series) & (df['Class'] == 1)].shape[0]
    flagged = df[flag_series].shape[0]
    total   = df[df['Class'] == 1].shape[0]
    recall    = caught / total * 100 if total else 0
    precision = caught / flagged * 100 if flagged else 0
    print(f"{label}")
    print(f"  fraud caught : {caught} / {total}")
    print(f"  total flagged: {flagged}")
    print(f"  recall       : {recall:.2f}%")
    print(f"  precision    : {precision:.2f}%")
    return recall, precision

## Attempt 1 — three engineered features only

My first instinct was to feed the model the features I had built by hand.

In [ ]:
from sklearn.ensemble import IsolationForest

features_v1 = df[['amount_zscore', 'hour', 'Amount']]

model_v1 = IsolationForest(contamination=0.0017, random_state=42)
model_v1.fit(features_v1)

df['ml_flagged_v1'] = model_v1.predict(features_v1)
evaluate(df['ml_flagged_v1'] == -1, 'Attempt 1 - 3 engineered features')

Recorded result: **1 fraud caught out of 492** — recall **0.20%**, precision **0.21%**.

Worse than the Class 3 rule and worse than the Class 2 z-score. Feeding a model only
three hand-made features starved it. `amount_zscore` is a deterministic transform of
`Amount`, so two of the three columns carry the same information.

## Attempt 2 — the full PCA feature set

The dataset's V1–V28 columns are PCA components of the original (confidential) transaction
features. They are where the real signal lives.

In [ ]:
feature_cols_v2 = [f'V{i}' for i in range(1, 29)] + ['Amount']

model_v2 = IsolationForest(contamination=0.0017, random_state=42)
model_v2.fit(df[feature_cols_v2])

df['ml_flagged_v2'] = model_v2.predict(df[feature_cols_v2])
evaluate(df['ml_flagged_v2'] == -1, 'Attempt 2 - V1..V28 + Amount')

## Attempt 3 — adding hour back in

This is the configuration that became `src/train_model.py` and ships in the scoring service.

In [ ]:
feature_cols_v3 = [f'V{i}' for i in range(1, 29)] + ['Amount', 'hour']

model_v3 = IsolationForest(
    n_estimators=200,
    contamination=0.0017,
    random_state=42,
    n_jobs=-1,
)
model_v3.fit(df[feature_cols_v3])

df['ml_flagged_v3'] = model_v3.predict(df[feature_cols_v3])
evaluate(df['ml_flagged_v3'] == -1, 'Attempt 3 - V1..V28 + Amount + hour (PRODUCTION)')

Recorded result for Attempt 3: **123 fraud caught out of 492** — recall **25%**, precision **25.4%**.

## The result that matters

| Detector | Features | Recall | Precision |
|---|---|---|---|
| Class 2 z-score | 1 | 2.2% | 0.27% |
| Class 3 combined rule | 3 | 9.96% | ~0.36% |
| Class 4 attempt 1 | 3 engineered | 0.20% | 0.21% |
| Class 4 attempt 3 | 30 | **25%** | **25.4%** |

Precision went from 0.27% to 25.4% — a **94x** improvement. Under the Class 3 rule I
needed ~275 false alarms per real fraud; now it's ~3. That is the difference between a
system nobody can use and one a business could actually run.

The jump did *not* come from switching to machine learning. Attempt 1 was machine
learning too, and it was the worst detector I built. It came from giving the model the
PCA components. **The features mattered more than the algorithm** — that is the single
most important thing I learned in this project.

## Persisting the model

The production version lives in `src/train_model.py` so the scoring service can load a
saved artifact instead of retraining on every request:

```bash
cd src
python train_model.py ../data/creditcard.csv   # writes model.joblib
```